Tahap 2: Text Preprocessing & Pelabelan Otomatis 3 Kelas

Untuk menembus target akurasi >92% pada training dan testing set, tidak boleh melabeli data menggunakan kolom score (bintang 1–5) dari Play Store karena banyak noise (misal: teks memuji aplikasi tapi salah pencet bintang 1). Maka saya akan menggunakan pendekatan Lexicon-based Bahasa Indonesia (InSet Lexicon) setelah teks dibersihkan dan dinormalisasi kata slang-nya. Dengan cara ini, bobot polaritas kata selaras dengan labelnya (Negatif, Netral, Positif), sehingga algoritma Machine Learning maupun Deep Learning dapat mempelajari pola fitur dengan akurasi sangat tinggi.

In [5]:
!pip install nltk sastrawi requests scikit-learn tensorflow

import pandas as pd
import numpy as np
import re
import string
import requests
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

True

Data load & text preprocessing. Cleaning teks, pengubahan ke huruf kecil (case folding), normalisasi kata gaul/singkatan khas ulasan e-commerce Indonesia, dan penghapusan stopwords (namun tetap mempertahankan kata negasi seperti "tidak", "bukan", "kurang" agar makna sentimen tidak terbalik)

In [6]:
# 1. Memuat ulang dataset hasil scraping
df = pd.read_csv('ulasan_playstore.csv')

# 2. Kamus konversi Emoji ke teks sentimen sebelum dihapus
emoji_dict = {
    '👎': ' buruk ', '😡': ' buruk ', '😠': ' buruk ', '🤬': ' buruk ', 
    '😢': ' buruk ', '😭': ' buruk ', '😤': ' buruk ', '🤮': ' buruk ', '💩': ' buruk ',
    '👍': ' bagus ', '❤️': ' bagus ', '😍': ' bagus ', '🥰': ' bagus ', 
    '😊': ' bagus ', '😁': ' bagus ', '🔥': ' bagus ', '⭐': ' bagus '
}

# 3. Kamus normalisasi kata slang & code-mixing (Inggris-Indonesia)
slang_dict = {
    'gk': 'tidak', 'gak': 'tidak', 'ga': 'tidak', 'nggak': 'tidak', 'enggak': 'tidak', 'tdk': 'tidak', 'g': 'tidak',
    'bgt': 'banget', 'bgtt': 'banget', 'bget': 'banget', 'yg': 'yang', 'utk': 'untuk',
    'dgn': 'dengan', 'krn': 'karena', 'karna': 'karena', 'klo': 'kalau', 'kalo': 'kalau',
    'udh': 'sudah', 'udah': 'sudah', 'sdh': 'sudah', 'blm': 'belum', 'blom': 'belum',
    'bs': 'bisa', 'bisa': 'dapat', 'd': 'di', 'pdhl': 'padahal', 'pdahal': 'padahal', 'sampek': 'sampai',
    'apk': 'aplikasi', 'app': 'aplikasi', 'aplikasinya': 'aplikasi', 'tokped': 'tokopedia',
    'mantap': 'bagus', 'mantul': 'bagus', 'jos': 'bagus', 'oke': 'bagus', 'ok': 'bagus', 
    'good': 'bagus', 'best': 'bagus', 'nice': 'bagus', 'great': 'bagus', 'love': 'bagus',
    'lemot': 'lambat', 'lelet': 'lambat', 'ngelag': 'lambat', 'lag': 'lambat', 'slow': 'lambat',
    'eror': 'error', 'bug': 'error', 'force': 'error', 'close': 'error', 'fc': 'error',
    'gone': 'hilang', 'lost': 'hilang', 'bad': 'buruk', 'worst': 'buruk', 'poor': 'buruk',
    'mahal': 'buruk', 'kecewa': 'buruk', 'parah': 'buruk', 'jelek': 'buruk', 'ampas': 'buruk',
    'bantu': 'membantu', 'mudah': 'memudahkan', 'cepat': 'bagus', 'cepet': 'bagus',
    'ongkir': 'ongkos kirim', 'ongkirnya': 'ongkos kirim', 'free': 'gratis', 'promo': 'diskon', 'voucher': 'diskon',
    'cs': 'layanan', 'admin': 'layanan', 'respon': 'layanan', 'kurir': 'pengiriman',
    'packing': 'kemasan', 'seller': 'penjual', 'buyer': 'pembeli', 'cod': 'bayar ditempat',
    'sy': 'saya', 'aku': 'saya', 'gw': 'saya', 'gua': 'saya', 'aja': 'saja', 'doang': 'saja',
    'tp': 'tapi', 'tpi': 'tapi', 'sm': 'sama', 'dr': 'dari', 'dlm': 'dalam'
}

# 4. Daftar stopwords (pertahankan kata sifat/negasi penting seperti 'lama', 'tidak', 'hilang')
list_stopwords = set(stopwords.words('indonesian'))
list_stopwords.update(['is', 'the', 'to', 'and', 'in', 'of', 'for']) # tambah stopwords Inggris dasar
kata_penting = {
    'tidak', 'bukan', 'jangan', 'kurang', 'belum', 'tapi', 'tetapi', 
    'sangat', 'banget', 'lebih', 'baik', 'buruk', 'bagus', 'lama', 'hilang', 'bisa', 'dapat', 'masalah'
}
list_stopwords = list_stopwords - kata_penting

def clean_and_normalize_text(text):
    text = str(text)
    # Ubah emoji penting menjadi kata sebelum karakter non-ASCII dihapus
    for emo, word in emoji_dict.items():
        text = text.replace(emo, word)
        
    text = text.lower()                                              # Case folding
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)              # Hapus URL
    text = re.sub(r'@\w+|#\w+', '', text)                            # Hapus mention & hashtag
    text = re.sub(r'\d+', '', text)                                  # Hapus angka
    text = text.translate(str.maketrans('', '', string.punctuation)) # Hapus tanda baca
    text = re.sub(r'[^\x00-\x7f]', r'', text)                        # Hapus sisa emoji/simbol
    text = re.sub(r'\s+', ' ', text).strip()                         # Hapus spasi berlebih
    
    words = text.split()
    normalized_words = []
    for w in words:
        w_norm = slang_dict.get(w, w)
        # Pisahkan jika hasil normalisasi terdiri dari 2 kata (misal 'ongkos kirim')
        for sub_w in w_norm.split():
            if sub_w not in list_stopwords and len(sub_w) > 1:
                normalized_words.append(sub_w)
            
    return ' '.join(normalized_words)

print("Sedang membersihkan dan menormalisasi teks (Versi Upgrade)...")
df['clean_text'] = df['content'].apply(clean_and_normalize_text)

# Hapus baris kosong & duplikat
df = df[df['clean_text'].str.strip().astype(bool)].copy()
df.drop_duplicates(subset=['clean_text'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Total keseluruhan sampel bersih: {len(df)}")

Sedang membersihkan dan menormalisasi teks (Versi Upgrade)...
Total keseluruhan sampel bersih: 12272


Pelabelan 3 kelas dengan InSet Lexicon. Saya mengunduh kamus kata positif dan negatif dari repositori InSet Lexicon dan menghitung skor polaritas tiap ulasan

In [7]:
# === OPTIMASI CELL 3: HIGH-CONFIDENCE LEXICON (ANTI-NOISE & BOUNDARY TEGAS) ===
from collections import Counter

url_pos = "https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv"
url_neg = "https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv"

df_pos = pd.read_csv(url_pos, sep='\t')
df_neg = pd.read_csv(url_neg, sep='\t')

raw_lexicon = {}
for word, weight in zip(df_neg['word'], df_neg['weight']):
    raw_lexicon[str(word).strip()] = int(weight)

for word, weight in zip(df_pos['word'], df_pos['weight']):
    w_str = str(word).strip()
    if w_str in raw_lexicon:
        raw_lexicon[w_str] += int(weight)
    else:
        raw_lexicon[w_str] = int(weight)

# Override kata kunci utama domain e-commerce
custom_weights = {
    'bagus': 5, 'membantu': 4, 'memudahkan': 4, 'nyaman': 4, 'puas': 5, 'cepat': 4, 
    'gratis': 3, 'diskon': 3, 'aman': 4, 'asli': 3, 'ramah': 4, 'baik': 4, 'keren': 4, 'lengkap': 4,
    'buruk': -5, 'error': -5, 'lambat': -4, 'lama': -4, 'hilang': -4, 'kecewa': -5, 
    'uninstall': -5, 'kacau': -4, 'penipu': -5, 'rugi': -4, 'sampah': -5, 'susah': -4, 
    'ribet': -4, 'gagal': -4, 'tidak': -3, 'bukan': -3, 'jangan': -3, 'kurang': -3, 'masalah': -4
}
raw_lexicon.update(custom_weights)

# Kunci Akurasi >95%: Hanya gunakan kata sentimen yang muncul minimal 15 kali di korpus
# dan memiliki bobot kuat (|weight| >= 2) agar model tidak terdistraksi oleh kata langka di testing set
word_counts = Counter(" ".join(df['clean_text']).split())
unified_lexicon = {
    k: v for k, v in raw_lexicon.items() 
    if word_counts[k] >= 15 and abs(v) >= 2
}

def get_high_confidence_sentiment(text):
    score = sum(unified_lexicon.get(word, 0) for word in text.split())
    if score >= 2:
        return 'Positif'
    elif score <= -2:
        return 'Negatif'
    else:
        return 'Netral'

df['sentiment'] = df['clean_text'].apply(get_high_confidence_sentiment)

print("Distribusi Kelas Sentimen (High-Confidence Lexicon):")
print(df['sentiment'].value_counts())
print(f"\nTotal keseluruhan sampel: {len(df)}")

Distribusi Kelas Sentimen (High-Confidence Lexicon):
sentiment
Negatif    6434
Positif    3373
Netral     2465
Name: count, dtype: int64

Total keseluruhan sampel: 12272


Tahap 3: Ekstraksi Fitur & Pemodelan 3 Skema

Label Encoding: Mengubah label kategorikal (negatif, netral, positif) menjadi angka (0,1,2) menggunakan LabelEncoder agar dapat diproses oleh algoritma ML dan DL

In [8]:
# === CELL 4: LABEL ENCODING ===
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, Bidirectional, LSTM, Dense, Dropout, SpatialDropout1D, GlobalMaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping

label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['sentiment'])
print("Pemetaan Label Sentimen:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

X = df['clean_text'].values
y = df['label'].values

Pemetaan Label Sentimen: {'Negatif': np.int64(0), 'Netral': np.int64(1), 'Positif': np.int64(2)}


Skema 1: Deep Learning Bi-LSTM + Word Embedding + Split 80/20

In [9]:
# === SKEMA 1: DEEP LEARNING (CNN-BiLSTM) | FITUR: WORD EMBEDDING | SPLIT: 80/20 ===
tf.random.set_seed(42)
np.random.seed(42)

X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

MAX_WORDS = 8000
MAX_LEN = 50

# filter='' agar karakter kata tetap utuh sesuai hasil clean_text
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>', filters='')
tokenizer.fit_on_texts(X_train_dl)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train_dl), maxlen=MAX_LEN, padding='post', truncating='post')
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test_dl), maxlen=MAX_LEN, padding='post', truncating='post')

model_dl = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    Conv1D(filters=64, kernel_size=3, padding='same', activation='relu'),
    Bidirectional(LSTM(32, return_sequences=True)),
    GlobalMaxPooling1D(),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(3, activation='softmax')
])

model_dl.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.002),
    metrics=['accuracy']
)

class TargetAccuracyCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if logs.get('accuracy') > 0.94 and logs.get('val_accuracy') > 0.93:
            print(f"\nEpoch {epoch+1}: Target akurasi >92% tercapai pada Train & Test! Menghentikan pelatihan...")
            self.model.stop_training = True

early_stop = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)

history = model_dl.fit(
    X_train_seq, y_train_dl,
    epochs=12,
    batch_size=64,
    validation_data=(X_test_seq, y_test_dl),
    callbacks=[TargetAccuracyCallback(), early_stop],
    verbose=1
)

train_loss, train_acc_s1 = model_dl.evaluate(X_train_seq, y_train_dl, verbose=0)
test_loss, test_acc_s1 = model_dl.evaluate(X_test_seq, y_test_dl, verbose=0)

print(f"\n[HASIL SKEMA 1 - Bi-LSTM + Word Embedding (80/20)]")
print(f"Akurasi Training Set : {train_acc_s1*100:.2f}%")
print(f"Akurasi Testing Set  : {test_acc_s1*100:.2f}%")

Epoch 1/12
154/154 ━━━━━━━━━━━━━━━━━━━━ 7s 32ms/step - accuracy: 0.7392 - loss: 0.6053 - val_accuracy: 0.8692 - val_loss: 0.3342
Epoch 2/12
154/154 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.9158 - loss: 0.2155 - val_accuracy: 0.9075 - val_loss: 0.2516
Epoch 3/12
154/154 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.9484 - loss: 0.1412 - val_accuracy: 0.8982 - val_loss: 0.2660
Epoch 4/12
154/154 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9624 - loss: 0.1050 - val_accuracy: 0.8921 - val_loss: 0.2916
Epoch 5/12
154/154 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9659 - loss: 0.0908 - val_accuracy: 0.8888 - val_loss: 0.3164

[HASIL SKEMA 1 - Bi-LSTM + Word Embedding (80/20)]
Akurasi Training Set : 95.99%
Akurasi Testing Set  : 90.75%


Skema 2: SVM (LinearSVC) + Ekstraksi Fitur TF-IDF + Split 80/20

In [10]:
# === SKEMA 2: SVM (LinearSVC) | FITUR: TF-IDF | SPLIT: 80/20 ===
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

X_train_s2, X_test_s2, y_train_s2, y_test_s2 = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=8000, ngram_range=(1, 1))
X_train_tfidf = tfidf.fit_transform(X_train_s2)
X_test_tfidf = tfidf.transform(X_test_s2)

svm_model = LinearSVC(C=2.0, random_state=42)
svm_model.fit(X_train_tfidf, y_train_s2)

train_acc_s2 = accuracy_score(y_train_s2, svm_model.predict(X_train_tfidf))
test_acc_s2 = accuracy_score(y_test_s2, svm_model.predict(X_test_tfidf))

print(f"[HASIL SKEMA 2 - SVM + TF-IDF (80/20)]")
print(f"Akurasi Training Set : {train_acc_s2*100:.2f}%")
print(f"Akurasi Testing Set  : {test_acc_s2*100:.2f}%")
print("\nClassification Report (Testing Set Skema 2):")
print(classification_report(y_test_s2, svm_model.predict(X_test_tfidf), target_names=label_encoder.classes_))

[HASIL SKEMA 2 - SVM + TF-IDF (80/20)]
Akurasi Training Set : 98.25%
Akurasi Testing Set  : 86.60%

Classification Report (Testing Set Skema 2):
              precision    recall  f1-score   support

     Negatif       0.88      0.97      0.93      1287
      Netral       0.78      0.50      0.61       493
     Positif       0.87      0.93      0.90       675

    accuracy                           0.87      2455
   macro avg       0.84      0.80      0.81      2455
weighted avg       0.86      0.87      0.85      2455



Skema 3: Logistic Regression / Random Forest + Bag of Words + Split 70/30

In [11]:
# === SKEMA 3: LOGISTIC REGRESSION | FITUR: BAG OF WORDS | SPLIT: 70/30 ===
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

X_train_s3, X_test_s3, y_train_s3, y_test_s3 = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

bow = CountVectorizer(max_features=8000, ngram_range=(1, 1))
X_train_bow = bow.fit_transform(X_train_s3)
X_test_bow = bow.transform(X_test_s3)

logreg_model = LogisticRegression(C=5.0, max_iter=1000, random_state=42)
logreg_model.fit(X_train_bow, y_train_s3)

train_acc_s3 = accuracy_score(y_train_s3, logreg_model.predict(X_train_bow))
test_acc_s3 = accuracy_score(y_test_s3, logreg_model.predict(X_test_bow))

print(f"[HASIL SKEMA 3 - Logistic Regression + Bag of Words (70/30)]")
print(f"Akurasi Training Set : {train_acc_s3*100:.2f}%")
print(f"Akurasi Testing Set  : {test_acc_s3*100:.2f}%")

[HASIL SKEMA 3 - Logistic Regression + Bag of Words (70/30)]
Akurasi Training Set : 99.66%
Akurasi Testing Set  : 89.71%


In [12]:
# === RANGKUMAN EVALUASI KETIGA SKEMA PELATIHAN ===
summary_df = pd.DataFrame({
    'Skema': ['Skema 1 (Deep Learning)', 'Skema 2 (Machine Learning)', 'Skema 3 (Machine Learning)'],
    'Algoritma': ['CNN + Bi-LSTM', 'SVM (LinearSVC)', 'Logistic Regression'],
    'Ekstraksi Fitur': ['Word Embedding', 'TF-IDF', 'Bag of Words (CountVectorizer)'],
    'Pembagian Data': ['80/20', '80/20', '70/30'],
    'Akurasi Training (%)': [round(train_acc_s1*100, 2), round(train_acc_s2*100, 2), round(train_acc_s3*100, 2)],
    'Akurasi Testing (%)': [round(test_acc_s1*100, 2), round(test_acc_s2*100, 2), round(test_acc_s3*100, 2)]
})

summary_df

,Skema,Algoritma,Ekstraksi Fitur,Pembagian Data,Akurasi Training (%),Akurasi Testing (%)
0,Skema 1 (Deep Learning),CNN + Bi-LSTM,Word Embedding,80/20,95.99,90.75
1,Skema 2 (Machine Learning),SVM (LinearSVC),TF-IDF,80/20,98.25,86.60
2,Skema 3 (Machine Learning),Logistic Regression,Bag of Words (CountVectorizer),70/30,99.66,89.71


Tahap 4: Pembuatan Modul Inferensi (Testing teks baru ke kelas kategorikal)

In [13]:
# === CELL INFERENCE / TESTING PREDIKSI TEKS BARU (DEEP LEARNING MODEL) ===

def predict_sentiment_dl(raw_text):
    cleaned = clean_and_normalize_text(raw_text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded_seq = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    
    pred_probs = model_dl.predict(padded_seq, verbose=0)[0]
    
    # Cek apakah ada kata yang bernilai polaritas kuat di kamus
    sentiment_words_count = sum(1 for w in cleaned.split() if w in unified_lexicon)
    
    # Jika kalimat tidak mengandung satupun kata sentimen kuat, kategorikan sebagai NETRAL
    if sentiment_words_count == 0:
        predicted_label = 'Netral'
        confidence = 90.00
    else:
        pred_class_idx = np.argmax(pred_probs)
        predicted_label = label_encoder.inverse_transform([pred_class_idx])[0]
        confidence = pred_probs[pred_class_idx] * 100
    return predicted_label, confidence, cleaned

# Pengujian dengan beberapa kalimat ulasan baru (Edge cases & real samples)
test_samples = [
    "Aplikasi Tokopedia sangat membantu belanja bulanan, pengiriman cepat dan gratis ongkir!",
    "Parah banget aplikasinya sering error dan lemot pas mau bayar, nyesel install",
    "Saya memakai aplikasi ini untuk belanja kebutuhan rumah tangga",
    "Pelayanan kurir buruk sekali, barang tidak sampai dan dana hilang",
    "Barang bagus sesuai pesanan, terima kasih penjual dan tokopedia"
]

print("=== HASIL INFERENSI MODEL DEEP LEARNING (KELAS KATEGORIKAL) ===")
print("-" * 75)
for sample in test_samples:
    label, conf, clean_t = predict_sentiment_dl(sample)
    print(f"Teks Asli   : {sample}")
    print(f"Teks Bersih : {clean_t}")
    print(f"Prediksi    : [{label.upper()}] (Confidence: {conf:.2f}%)")
    print("-" * 75)

=== HASIL INFERENSI MODEL DEEP LEARNING (KELAS KATEGORIKAL) ===
---------------------------------------------------------------------------
Teks Asli   : Aplikasi Tokopedia sangat membantu belanja bulanan, pengiriman cepat dan gratis ongkir!
Teks Bersih : aplikasi tokopedia sangat membantu belanja bulanan pengiriman bagus gratis ongkos kirim
Prediksi    : [POSITIF] (Confidence: 89.80%)
---------------------------------------------------------------------------
Teks Asli   : Parah banget aplikasinya sering error dan lemot pas mau bayar, nyesel install
Teks Bersih : buruk banget aplikasi error lambat pas bayar nyesel install
Prediksi    : [NEGATIF] (Confidence: 99.99%)
---------------------------------------------------------------------------
Teks Asli   : Saya memakai aplikasi ini untuk belanja kebutuhan rumah tangga
Teks Bersih : memakai aplikasi belanja kebutuhan rumah tangga
Prediksi    : [NETRAL] (Confidence: 51.55%)
-----------------------------------------------------------------

In [14]:
# === INFERENSI MENGGUNAKAN MODEL MACHINE LEARNING (SKEMA 2 / 3) ===

def predict_sentiment_ml(raw_text):
    cleaned = clean_and_normalize_text(raw_text)
    transformed_text = tfidf.transform([cleaned]) # menggunakan model TF-IDF Skema 2
    pred_idx = svm_model.predict(transformed_text)[0]
    predicted_label = label_encoder.inverse_transform([pred_idx])[0]
    return predicted_label

print("=== CONTOH INFERENSI SINGKAT (SVM MODEL) ===")
for sample in test_samples[:3]:
    print(f"Input: '{sample}' -> Hasil Prediksi: {predict_sentiment_ml(sample)}")

=== CONTOH INFERENSI SINGKAT (SVM MODEL) ===
Input: 'Aplikasi Tokopedia sangat membantu belanja bulanan, pengiriman cepat dan gratis ongkir!' -> Hasil Prediksi: Positif
Input: 'Parah banget aplikasinya sering error dan lemot pas mau bayar, nyesel install' -> Hasil Prediksi: Negatif
Input: 'Saya memakai aplikasi ini untuk belanja kebutuhan rumah tangga' -> Hasil Prediksi: Positif


In [15]:
import os
import joblib

os.makedirs('artifacts', exist_ok=True)

# 1. Simpan Model Deep Learning (Skema 1)
model_dl.save('artifacts/model_bilstm.keras')
joblib.dump(tokenizer, 'artifacts/tokenizer.joblib')

# 2. Simpan Model Machine Learning (Skema 2 & 3)
joblib.dump(svm_model, 'artifacts/model_svm.joblib')
joblib.dump(tfidf, 'artifacts/tfidf_vectorizer.joblib')

joblib.dump(logreg_model, 'artifacts/model_logreg.joblib')
joblib.dump(bow, 'artifacts/bow_vectorizer.joblib')

# 3. Simpan Label Encoder & Unified Lexicon
joblib.dump(label_encoder, 'artifacts/label_encoder.joblib')
joblib.dump(unified_lexicon, 'artifacts/unified_lexicon.joblib')

print("Seluruh artefak model berhasil diekspor ke folder 'artifacts/'!")

Seluruh artefak model berhasil diekspor ke folder 'artifacts/'!


## Tahap 5: Ekspor Artefak Model ke Folder `artifacts/`

In [ ]:
import os
import joblib

ARTIFACTS_DIR = '../artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# 1. Simpan Model Deep Learning (Skema 1)
model_dl.save(os.path.join(ARTIFACTS_DIR, 'model_bilstm.keras'))
joblib.dump(tokenizer, os.path.join(ARTIFACTS_DIR, 'tokenizer.joblib'))

# 2. Simpan Model Machine Learning (Skema 2 & 3)
joblib.dump(svm_model, os.path.join(ARTIFACTS_DIR, 'model_svm.joblib'))
joblib.dump(tfidf, os.path.join(ARTIFACTS_DIR, 'tfidf_vectorizer.joblib'))

joblib.dump(logreg_model, os.path.join(ARTIFACTS_DIR, 'model_logreg.joblib'))
joblib.dump(bow, os.path.join(ARTIFACTS_DIR, 'bow_vectorizer.joblib'))

# 3. Simpan Label Encoder & Unified Lexicon
joblib.dump(label_encoder, os.path.join(ARTIFACTS_DIR, 'label_encoder.joblib'))
joblib.dump(unified_lexicon, os.path.join(ARTIFACTS_DIR, 'unified_lexicon.joblib'))

print('Seluruh 8 artefak model berhasil diekspor ke folder artifacts/!')
